# Merge sensor logger data

October 2025

Merge sensor loggeer data into one sqlite db

- Source data are sqlite exports from the sensor logger app

- In the `./data` folder

- Output file is `./data/karting_merged.sqlite`

Filter data for the race and label location records

- Filter for when location crosses start/finish liune for reach race. 

- Only keep the data between first time it crosses and last time it crosses for each race.

- Add sector number to location based on the sector linestrings in the config

## Merge Karting info

Update the following configuration json and then run the rest of the notebook

In [2]:
# config info about the circuit, metadata, etc.
race_config = [
    {
        "filename": "./data/gateway-1.sqlite",
        "race_date": "2025-10-02",
        "proskill_start": 0,
        "proskill_end": 0,
        "proskill_delta": 0,
        "circuit": "Gateway Kartplex",
        "sf_line": "LineString([(38.64886, -90.13461), (38.64888, -90.13440)])",
        "s1_s2_line": "LineString([(38.64918, -90.13474), (38.64911, -90.13491)])",
        "s2_s3_line": "LineString([(38.64868, -90.13454), (38.64852, -90.13473)])",
        "visit": 1,
        "race": 1
    },
    {
        "filename": "./data/gateway-2.sqlite",
        "race_date": "2025-10-02",
        "proskill_start": 0,
        "proskill_end": 0,
        "proskill_delta": 0,
        "circuit": "Gateway Kartplex",
        "sf_line": "LineString([(38.64886, -90.13461), (38.64888, -90.13440)])",
        "s1_s2_line": "LineString([(38.64918, -90.13474), (38.64911, -90.13491)])",
        "s2_s3_line": "LineString([(38.64868, -90.13454), (38.64852, -90.13473)])",
        "visit": 1,
        "race": 2
    },
    {
        "filename": "./data/gateway-3.sqlite",
        "race_date": "2025-10-02",
        "proskill_start": 0,
        "proskill_end": 0,
        "proskill_delta": 0,
        "circuit": "Gateway Kartplex",
        "sf_line": "LineString([(38.64886, -90.13461), (38.64888, -90.13440)])",
        "s1_s2_line": "LineString([(38.64918, -90.13474), (38.64911, -90.13491)])",
        "s2_s3_line": "LineString([(38.64868, -90.13454), (38.64852, -90.13473)])",
        "visit": 1,
        "race": 3
    },
    {
        "filename": "./data/gateway-4.sqlite",
        "race_date": "2025-10-02",
        "proskill_start": 0,
        "proskill_end": 0,
        "proskill_delta": 0,
        "circuit": "Gateway Kartplex",
        "sf_line": "LineString([(38.64886, -90.13461), (38.64888, -90.13440)])",
        "s1_s2_line": "LineString([(38.64918, -90.13474), (38.64911, -90.13491)])",
        "s2_s3_line": "LineString([(38.64868, -90.13454), (38.64852, -90.13473)])",
        "visit": 1,
        "race": 4
    },
    {
        "filename": "./data/700_IL-203_S-2025-10-16_22-04-11.sqlite",
        "race_date": "2025-10-16",
        "proskill_start": 0,
        "proskill_end": 0,
        "proskill_delta": 0,
        "circuit": "Gateway Kartplex",
        "sf_line": "LineString([(38.64886, -90.13461), (38.64888, -90.13440)])",
        "s1_s2_line": "LineString([(38.64918, -90.13474), (38.64911, -90.13491)])",
        "s2_s3_line": "LineString([(38.64868, -90.13454), (38.64852, -90.13473)])",
        "visit": 2,
        "race": 5
    },
    {
        "filename": "./data/700_IL-203_S-2025-10-16_22-31-59.sqlite",
        "race_date": "2025-10-16",
        "proskill_start": 0,
        "proskill_end": 0,
        "proskill_delta": 0,
        "circuit": "Gateway Kartplex",
        "sf_line": "LineString([(38.64886, -90.13461), (38.64888, -90.13440)])",
        "s1_s2_line": "LineString([(38.64918, -90.13474), (38.64911, -90.13491)])",
        "s2_s3_line": "LineString([(38.64868, -90.13454), (38.64852, -90.13473)])",
        "visit": 2,
        "race": 6
    },
    {
        "filename": "./data/700_IL-203_S-2025-10-16_22-50-34.sqlite",
        "race_date": "2025-10-16",
        "proskill_start": 0,
        "proskill_end": 0,
        "proskill_delta": 0,
        "circuit": "Gateway Kartplex",
        "sf_line": "LineString([(38.64886, -90.13461), (38.64888, -90.13440)])",
        "s1_s2_line": "LineString([(38.64918, -90.13474), (38.64911, -90.13491)])",
        "s2_s3_line": "LineString([(38.64868, -90.13454), (38.64852, -90.13473)])",
        "visit": 2,
        "race": 7
    }
]


## Merge and enrich

Add metadata to data set based on config above

Note the `exclude_tables = ['Bluetooth','WatchMagnetometer', 'WatchBarometer', 'WatchLocation', 'WristMotion']` to account for settings that haven't always been on and would cause errors when building out the db


In [3]:
import sqlite3
import pandas as pd

# Create merged database
merged_conn = sqlite3.connect('./data/karting_merged.sqlite')

# Was recording watch but it wasn't working well for analysis
exclude_tables = ['Bluetooth','WatchMagnetometer', 'WatchBarometer', 'WatchLocation', 'WristMotion']

# Create RaceMetadata table
race_metadata = pd.DataFrame([{
    'race': g['race'],
    'visit': g['visit'],
    'race_date': g['race_date'],
    'circuit': g['circuit'],
    'proskill_start': g['proskill_start'],
    'proskill_end': g['proskill_end'],
    'proskill_delta': g['proskill_delta']
} for g in race_config])

race_metadata.to_sql('RaceMetadata', merged_conn, if_exists='replace', index=False)


# Merge all databases
for gateway in race_config:
    source_conn = sqlite3.connect(gateway['filename'])
    
    # Get all table names from source database
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", source_conn)
    
    for table_name in tables['name']:
        if table_name in exclude_tables:
            continue        
        
        df = pd.read_sql_query(f"SELECT * FROM {table_name}", source_conn)
        df['race'] = gateway['race']
        df['visit'] = gateway['visit']
        df.to_sql(table_name, merged_conn, if_exists='append', index=False)
        
    source_conn.close()
    print(f"Merged {gateway['filename']} - Race {gateway['race']}")

merged_conn.close()
print("\nMerge complete: ./data/karting_merged.sqlite")

Merged ./data/gateway-1.sqlite - Race 1
Merged ./data/gateway-2.sqlite - Race 2
Merged ./data/gateway-3.sqlite - Race 3
Merged ./data/gateway-4.sqlite - Race 4
Merged ./data/700_IL-203_S-2025-10-16_22-04-11.sqlite - Race 5
Merged ./data/700_IL-203_S-2025-10-16_22-31-59.sqlite - Race 6
Merged ./data/700_IL-203_S-2025-10-16_22-50-34.sqlite - Race 7

Merge complete: ./data/karting_merged.sqlite


## check results

Some basic checks on the data

In [8]:
import sqlite3
import pandas as pd
from shapely.geometry import Point, LineString

conn = sqlite3.connect('./data/karting_merged.sqlite')

# Load Location data
location = pd.read_sql_query("SELECT * FROM Location ORDER BY race, time", conn)

# Add lap_number column
location['lap_number'] = 0

for gateway in raw_gateway:
    race_id = gateway['race']
    
    # Parse start/finish line
    sf_line_coords = eval(gateway['sf_line'].replace('LineString', ''))
    sf_line = LineString(sf_line_coords)
    
    # Get data for this race
    race_mask = location['race'] == race_id
    race_data = location[race_mask].copy().reset_index(drop=True)
    
    # Track line crossings
    crossings = []
    prev_point = None
    
    for i, row in race_data.iterrows():
        current_point = Point(row['latitude'], row['longitude'])
        
        if prev_point is not None:
            segment = LineString([prev_point, current_point])
            if segment.intersects(sf_line):
                crossings.append(i)
        
        prev_point = current_point
    
    if len(crossings) > 0:
        # Mark rows to keep (between first and last crossing)
        first_crossing = crossings[0]
        last_crossing = crossings[-1]
        
        # Update lap numbers for kept rows
        lap_num = 1
        for i in range(len(crossings)):
            if i < len(crossings) - 1:
                start_idx = crossings[i]
                end_idx = crossings[i + 1]
                race_data.loc[start_idx:end_idx-1, 'lap_number'] = lap_num
                lap_num += 1
        
        # Keep only data between first and last crossing
        race_data = race_data.loc[first_crossing:last_crossing]
        
        # Update main dataframe
        location.loc[race_mask, :] = race_data
        location = location[~((location['race'] == race_id) & (location['lap_number'] == 0))]
        
        print(f"Race {race_id}: {len(crossings)} crossings, {lap_num-1} laps")

# Save updated Location table
location.to_sql('Location', conn, if_exists='replace', index=False)

conn.close()
print("\nLocation table updated with lap numbers")

NameError: name 'raw_gateway' is not defined

## Filter data

- Here we use the start/finish line to filter the sensor data for each race.

- For each race, only keep the first time crossing the start finish, and the last time crossing the start finish.

In [5]:
# identify location column names

import sqlite3
import pandas as pd

conn = sqlite3.connect('./data/karting_merged.sqlite')

location = pd.read_sql_query("SELECT * FROM Location LIMIT 1", conn)
print("Location table columns:")
print(location.columns.tolist())

conn.close()

Location table columns:
['time', 'seconds_elapsed', 'altitude', 'speedAccuracy', 'bearingAccuracy', 'latitude', 'altitudeAboveMeanSeaLevel', 'bearing', 'horizontalAccuracy', 'verticalAccuracy', 'longitude', 'speed', 'utc_time', 'race', 'visit']


## Label laps and remove data before first sf_line and after last sf_line crossing

In [10]:
import sqlite3
import pandas as pd
from shapely.geometry import Point, LineString

conn = sqlite3.connect('./data/karting_merged.sqlite')

# Load Location data
location = pd.read_sql_query("SELECT * FROM Location ORDER BY race, time", conn)

# Add lap_number column
location['lap_number'] = 0

# Track indices to delete
indices_to_delete = []

for race in race_config:
    race_id = race['race']
    
    # Parse start/finish line
    sf_line_coords = eval(race['sf_line'].replace('LineString', ''))
    sf_line = LineString(sf_line_coords)
    
    # Get data for this race
    race_indices = location[location['race'] == race_id].index
    race_data = location.loc[race_indices].copy()
    
    # Track line crossings
    crossings = []
    prev_point = None
    
    for idx in race_indices:
        row = location.loc[idx]
        current_point = Point(row['latitude'], row['longitude'])
        
        if prev_point is not None:
            segment = LineString([prev_point, current_point])
            if segment.intersects(sf_line):
                crossings.append(idx)
        
        prev_point = current_point
    
    if len(crossings) > 0:
        # Mark data before first crossing for deletion
        first_crossing_idx = crossings[0]
        indices_to_delete.extend(race_indices[race_indices < first_crossing_idx])
        
        # Mark data after last crossing for deletion
        last_crossing_idx = crossings[-1]
        indices_to_delete.extend(race_indices[race_indices > last_crossing_idx])
        
        # Assign lap numbers between crossings
        lap_num = 1
        for i in range(len(crossings) - 1):
            start_idx = crossings[i]
            end_idx = crossings[i + 1]
            lap_indices = race_indices[(race_indices >= start_idx) & (race_indices < end_idx)]
            location.loc[lap_indices, 'lap_number'] = lap_num
            lap_num += 1
        
        print(f"Race {race_id}: {len(crossings)} crossings, {lap_num-1} laps")

# Delete marked rows
location = location.drop(indices_to_delete)

# Save updated Location table
location.to_sql('Location', conn, if_exists='replace', index=False)

conn.close()
print(f"\nDeleted {len(indices_to_delete)} rows")
print("Location table updated with lap numbers")


Race 1: 13 crossings, 12 laps
Race 2: 12 crossings, 11 laps
Race 3: 13 crossings, 12 laps
Race 4: 13 crossings, 12 laps
Race 5: 14 crossings, 13 laps
Race 6: 14 crossings, 13 laps
Race 7: 14 crossings, 13 laps

Deleted 1746 rows
Location table updated with lap numbers


In [11]:
import sqlite3
import pandas as pd

# Create merged database
merged_conn = sqlite3.connect('./data/karting_merged.sqlite')

# Was recording watch but it wasn't working well for analysis
exclude_tables = ['Bluetooth','WatchMagnetometer', 'WatchBarometer', 'WatchLocation', 'WristMotion']

# Create RaceMetadata table
race_metadata = pd.DataFrame([{
    'race': g['race'],
    'visit': g['visit'],
    'race_date': g['race_date'],
    'circuit': g['circuit'],
    'proskill_start': g['proskill_start'],
    'proskill_end': g['proskill_end'],
    'proskill_delta': g['proskill_delta']
} for g in race_config])

race_metadata.to_sql('RaceMetadata', merged_conn, if_exists='replace', index=False)

# Merge all databases
for gateway in race_config:
    source_conn = sqlite3.connect(gateway['filename'])
    
    # Get all table names from source database
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", source_conn)
    
    for table_name in tables['name']:
        if table_name in exclude_tables:
            continue        
        
        df = pd.read_sql_query(f"SELECT * FROM {table_name}", source_conn)
        df['race'] = gateway['race']
        df['visit'] = gateway['visit']
        df.to_sql(table_name, merged_conn, if_exists='append', index=False)
        
    source_conn.close()
    print(f"Merged {gateway['filename']} - Race {gateway['race']}")

# Print columns from the Location table after merging
location_df = pd.read_sql_query("SELECT * FROM Location LIMIT 1", merged_conn)
print("Location table columns:", list(location_df.columns))

merged_conn.close()
print("\nMerge complete: ./data/karting_merged.sqlite")

Merged ./data/gateway-1.sqlite - Race 1
Merged ./data/gateway-2.sqlite - Race 2
Merged ./data/gateway-3.sqlite - Race 3
Merged ./data/gateway-4.sqlite - Race 4
Merged ./data/700_IL-203_S-2025-10-16_22-04-11.sqlite - Race 5
Merged ./data/700_IL-203_S-2025-10-16_22-31-59.sqlite - Race 6
Merged ./data/700_IL-203_S-2025-10-16_22-50-34.sqlite - Race 7
Location table columns: ['time', 'seconds_elapsed', 'altitude', 'speedAccuracy', 'bearingAccuracy', 'latitude', 'altitudeAboveMeanSeaLevel', 'bearing', 'horizontalAccuracy', 'verticalAccuracy', 'longitude', 'speed', 'utc_time', 'race', 'visit', 'lap_number']

Merge complete: ./data/karting_merged.sqlite
